# Recidivism model comparison

Compare CatBoost, Explainable Boosting Machine (EBM), XGBoost, LightGBM, and a stacked ensemble on the NIJ workbook. The target is recorded recidivism within three years.

Run all cells with a Python 3 kernel using the packages in the repository's `requirements.txt` plus Jupyter. Start from the repository root or the `ml_pipeline` folder.

Use the workbook's `Training_Sample` flag for the development/holdout boundary. Inside the development sample, fit candidates on a stratified subset, then choose model settings and probability thresholds by validation accuracy. Refit the selected settings on all development rows. The workbook holdout is used only in the final evaluation. Earlier repo experiments used random splits of the same workbook, so this is a holdout for this workflow rather than an independent external dataset.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from interpret.glassbox import ExplainableBoostingClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import StackingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

SEED = 42
THREADS = 2
TARGET = "Recidivism_Within_3years"
DATA = Path("nijdataset.xlsx")
if not DATA.is_file():
    DATA = Path("ml_pipeline") / DATA

## Data and feature boundary

Use the 28 predictors shared by the existing baseline scripts. The allowlist excludes the participant ID, split flag, later supervision activity, and all recidivism outcomes. Category normalization is row-wise. Median imputation and one-hot categories are learned inside each model fit.

In [2]:
FEATURES = [
    "Gender",
    "Race",
    "Age_at_Release",
    "Residence_PUMA",
    "Gang_Affiliated",
    "Supervision_Risk_Score_First",
    "Supervision_Level_First",
    "Education_Level",
    "Dependents",
    "Prison_Offense",
    "Prison_Years",
    "Prior_Arrest_Episodes_Felony",
    "Prior_Arrest_Episodes_Misd",
    "Prior_Arrest_Episodes_Violent",
    "Prior_Arrest_Episodes_Property",
    "Prior_Arrest_Episodes_Drug",
    "Prior_Arrest_Episodes_DVCharges",
    "Prior_Arrest_Episodes_GunCharges",
    "Prior_Conviction_Episodes_Felony",
    "Prior_Conviction_Episodes_Misd",
    "Prior_Conviction_Episodes_Viol",
    "Prior_Conviction_Episodes_Prop",
    "Prior_Conviction_Episodes_Drug",
    "Prior_Revocations_Parole",
    "Prior_Revocations_Probation",
    "Condition_MH_SA",
    "Condition_Cog_Ed",
    "Condition_Other",
]

numeric = ["Residence_PUMA", "Supervision_Risk_Score_First"]
categorical = [column for column in FEATURES if column not in numeric]

data = pd.read_excel(DATA, usecols=[*FEATURES, TARGET, "Training_Sample"])
y = data[TARGET].map({"No": 0, "Yes": 1})
if y.isna().any() or set(data["Training_Sample"].unique()) != {0, 1}:
    raise ValueError("Unexpected recidivism labels or Training_Sample flags.")
y = y.astype(int)

X = data[FEATURES].copy()
for column in categorical:
    X[column] = X[column].fillna("Missing").astype(str)
for column in numeric:
    X[column] = pd.to_numeric(X[column], errors="coerce")

development = data["Training_Sample"].eq(1)
X_dev, y_dev = X.loc[development], y.loc[development]
X_fit, X_val, y_fit, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED,
)
print(f"Fit: {len(y_fit):,} | Validation: {len(y_val):,} | Holdout: {(~development).sum():,}")

Fit: 13,521 | Validation: 4,507 | Holdout: 7,807


## Baselines and tuning candidates

The first configuration for each model reproduces the original baseline settings. Additional configurations test different tree depths, learning rates, regularization, or interaction counts. The stacking model learns logistic weights from five-fold out-of-fold predictions of XGBoost, LightGBM, and CatBoost. Preprocessing runs inside those folds as part of each base pipeline.

In [3]:
preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ("numeric", SimpleImputer(strategy="median"), numeric),
])

def pipeline(estimator):
    return make_pipeline(clone(preprocessor), estimator)

def xgboost(**settings):
    return pipeline(XGBClassifier(
        random_state=SEED, n_jobs=THREADS, eval_metric="logloss", **settings,
    ))

def lightgbm(**settings):
    return pipeline(LGBMClassifier(
        random_state=SEED, n_jobs=THREADS, verbosity=-1, **settings,
    ))

def catboost(**settings):
    return pipeline(CatBoostClassifier(
        random_seed=SEED, thread_count=THREADS, verbose=False,
        allow_writing_files=False, **settings,
    ))

def ebm(**settings):
    return pipeline(ExplainableBoostingClassifier(
        random_state=SEED, n_jobs=THREADS, **settings,
    ))

def stack(xgb, lgbm, cat):
    return StackingClassifier(
        estimators=[("xgb", clone(xgb)), ("lgbm", clone(lgbm)), ("cat", clone(cat))],
        final_estimator=LogisticRegression(random_state=SEED),
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        stack_method="predict_proba",
    )

candidates = {
    "CatBoost": {
        "Original": catboost(iterations=100, learning_rate=0.10, depth=6),
        "Shallow": catboost(iterations=750, learning_rate=0.035, depth=4),
        "Regularized": catboost(iterations=550, learning_rate=0.04, depth=6, l2_leaf_reg=8),
    },
    "EBM": {
        "Original": ebm(),
        "Additive": ebm(interactions=0),
        "10 interactions": ebm(interactions=10),
    },
    "LightGBM": {
        "Original": lightgbm(n_estimators=100, learning_rate=0.10),
        "15 leaves": lightgbm(n_estimators=450, learning_rate=0.035, num_leaves=15,
                              min_child_samples=45, reg_lambda=3),
        "7 leaves": lightgbm(n_estimators=700, learning_rate=0.03, num_leaves=7,
                             min_child_samples=60, reg_lambda=3),
    },
    "XGBoost": {
        "Original": xgboost(n_estimators=100, learning_rate=0.10, max_depth=5),
        "Depth 3": xgboost(n_estimators=550, learning_rate=0.035, max_depth=3,
                           min_child_weight=10, subsample=0.85, colsample_bytree=0.85),
        "Depth 4": xgboost(n_estimators=425, learning_rate=0.04, max_depth=4,
                           min_child_weight=12, reg_lambda=5, subsample=0.85),
    },
}

## Select on validation accuracy

Search thresholds from 0.30 through 0.70 in steps of 0.005. Break threshold ties by proximity to 0.50 and candidate ties by validation ROC-AUC. The comparison keeps each original model at the standard 0.50 cutoff as a reference.

In [4]:
threshold_grid = sorted(np.linspace(0.30, 0.70, 81), key=lambda value: abs(value - 0.50))

def tune(configurations):
    trials = []
    for setting, estimator in configurations.items():
        probabilities = clone(estimator).fit(X_fit, y_fit).predict_proba(X_val)[:, 1]
        threshold = max(threshold_grid, key=lambda t: accuracy_score(y_val, probabilities >= t))
        trials.append({
            "Configuration": setting,
            "Threshold": threshold,
            "Validation accuracy": accuracy_score(y_val, probabilities >= threshold),
            "Validation ROC-AUC": roc_auc_score(y_val, probabilities),
        })
    return max(trials, key=lambda row: (row["Validation accuracy"], row["Validation ROC-AUC"]))

selected = {}
for family, configurations in candidates.items():
    selected[family] = tune(configurations)
    print(f"{family}: {selected[family]['Configuration']}", flush=True)

boosters = ("XGBoost", "LightGBM", "CatBoost")
candidates["Stacked ensemble"] = {
    "Original": stack(*(candidates[name]["Original"] for name in boosters)),
    "Selected boosters": stack(*(
        candidates[name][selected[name]["Configuration"]] for name in boosters
    )),
}
selected["Stacked ensemble"] = tune(candidates["Stacked ensemble"])

champion = max(
    selected, key=lambda name: (selected[name]["Validation accuracy"],
                                selected[name]["Validation ROC-AUC"]),
)
display(pd.DataFrame.from_dict(selected, orient="index").round(4))
print(f"Selected model: {champion}")

CatBoost: Shallow
EBM: 10 interactions
LightGBM: 7 leaves
XGBoost: Depth 3
                      Configuration  Threshold  Validation accuracy  Validation ROC-AUC
CatBoost                    Shallow      0.480               0.6843              0.7285
EBM                 10 interactions      0.510               0.6874              0.7309
LightGBM                   7 leaves      0.475               0.6812              0.7255
XGBoost                     Depth 3      0.475               0.6843              0.7282
Stacked ensemble  Selected boosters      0.475               0.6840              0.7286
Selected model: EBM


## Compare all models on the same holdout

The tuned rows use the thresholds chosen on validation. Accuracy gains are paired differences against each family's original configuration on these same records. The selected model was fixed before this comparison.

In [5]:
X_test, y_test = X.loc[~development], y.loc[~development]
fitted = {}
predictions = {}
rows = []
for family, configurations in candidates.items():
    choice = selected[family]
    for label, setting, threshold in [
        ("Original", "Original", 0.50),
        ("Tuned", choice["Configuration"], choice["Threshold"]),
    ]:
        key = (family, setting)
        if key not in fitted:
            fitted[key] = clone(configurations[setting]).fit(X_dev, y_dev)
        probabilities = fitted[key].predict_proba(X_test)[:, 1]
        predicted = probabilities >= threshold
        predictions[family, label] = predicted
        rows.append({
            "Model": family, "Version": label, "Configuration": setting,
            "Threshold": threshold, "Accuracy": accuracy_score(y_test, predicted),
            "Balanced accuracy": balanced_accuracy_score(y_test, predicted),
            "Precision": precision_score(y_test, predicted, zero_division=0),
            "Recall": recall_score(y_test, predicted, zero_division=0),
            "F1": f1_score(y_test, predicted, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, probabilities),
        })

results = pd.DataFrame(rows).set_index(["Model", "Version"])
original = results.xs("Original", level="Version")["Accuracy"]
tuned = results.xs("Tuned", level="Version")["Accuracy"]
display(results.round(4))
display((100 * (tuned - original)).rename("Accuracy change (percentage points)").round(2))
print(f"Majority-class reference: {accuracy_score(y_test, np.repeat(y_dev.mode().iat[0], len(y_test))):.2%}")
print(f"{champion} holdout accuracy: {tuned[champion]:.2%} "
      f"(original {original[champion]:.2%})")
display(pd.DataFrame(
    confusion_matrix(y_test, predictions[champion, "Tuned"], labels=[0, 1]),
    index=["Actual no", "Actual yes"],
    columns=["Predicted no", "Predicted yes"],
))

                               Configuration  Threshold  Accuracy  Balanced accuracy  Precision  Recall      F1  ROC-AUC
Model            Version                                                                                                
CatBoost         Original           Original      0.500    0.6799             0.6542     0.6826  0.8273  0.7480   0.7287
                 Tuned               Shallow      0.480    0.6785             0.6503     0.6775  0.8401  0.7500   0.7311
EBM              Original           Original      0.500    0.6780             0.6553     0.6866  0.8082  0.7424   0.7282
                 Tuned       10 interactions      0.510    0.6831             0.6621     0.6934  0.8035  0.7444   0.7319
LightGBM         Original           Original      0.500    0.6734             0.6500     0.6822  0.8073  0.7395   0.7251
                 Tuned              7 leaves      0.475    0.6813             0.6527     0.6787  0.8452  0.7528   0.7287
XGBoost          Original       

## Export the selected model

Set `EXPORT_MODEL` to `True` to save the fitted model, input schema, and selected threshold in one bundle. For inference, apply the same row-wise category and numeric normalization as above, then use `predict_proba` and the saved threshold.

In [6]:
EXPORT_MODEL = False
if EXPORT_MODEL:
    choice = selected[champion]
    destination = DATA.parent / "saved_models" / "release_risk_champion.pkl"
    destination.parent.mkdir(exist_ok=True)
    joblib.dump({
        "model_name": champion,
        "model": fitted[champion, choice["Configuration"]],
        "threshold": float(choice["Threshold"]),
        "features": FEATURES,
        "categorical_features": categorical,
        "numeric_features": numeric,
        "target": TARGET,
    }, destination)
    print(f"Saved {destination}")

## Accuracy improvement experiments

Keep the same prediction point and workbook split for a direct comparison with the 68.31% EBM reference above. Restore the four prior-history fields identified in the repository's champion script, preserve missingness, encode ordered ranges, and summarize prior history. Use the workbook rows as recorded. Bootstrap-based models resample or reweight training records internally.

Select every configuration and cutoff using the original development validation split. The workbook holdout was already reported in the first experiment, so the follow-up scores are a benchmark on a reused holdout. Independent data would be needed for a fresh estimate.

In [7]:
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

prior_aliases = {
    "_v1": "Prior_Arrest_Episodes_PPViolationCharges",
    "_v2": "Prior_Conviction_Episodes_PPViolationCharges",
    "_v3": "Prior_Conviction_Episodes_DomesticViolenceCharges",
    "_v4": "Prior_Conviction_Episodes_GunCharges",
}
history = pd.read_excel(DATA, usecols=list(prior_aliases)).rename(columns=prior_aliases)
release = pd.concat([X, history], axis=1)

ordered_counts = [
    column for column in release
    if column.startswith(("Prior_Arrest_Episodes_", "Prior_Conviction_Episodes_"))
    and not release[column].isin(["Yes", "No"]).all()
]
for column in [*ordered_counts, "Dependents"]:
    # Top-coded categories retain their lower bound as an ordinal value.
    release[column] = pd.to_numeric(
        release[column].astype(str).str.extract(r"^(\d+)", expand=False), errors="coerce",
    )

release["Age_at_Release"] = pd.to_numeric(
    release["Age_at_Release"].str.extract(r"^(\d+)", expand=False), errors="coerce",
)
release["Prison_Years"] = release["Prison_Years"].map({
    "Less than 1 year": 0, "1-2 years": 1,
    "Greater than 2 to 3 years": 2, "More than 3 years": 3,
})
release["Education_Level"] = release["Education_Level"].map({
    "Less than HS diploma": 0, "High School Diploma": 1, "At least some college": 2,
})
release["Residence_PUMA"] = release["Residence_PUMA"].astype(str)
for prefix, summary in [
    ("Prior_Arrest_Episodes_", "arrest_history"),
    ("Prior_Conviction_Episodes_", "conviction_history"),
]:
    counts = release[[column for column in ordered_counts if column.startswith(prefix)]]
    release[f"{summary}_breadth"] = counts.gt(0).sum(axis=1)
    release[f"{summary}_max"] = counts.max(axis=1)
    # The categories may overlap, so this is a feature score rather than a count of distinct events.
    release[f"{summary}_score"] = counts.sum(axis=1, min_count=1)

release_categorical = list(release.select_dtypes(exclude="number").columns)
release_numeric = [column for column in release if column not in release_categorical]
for column in release_categorical:
    release[column] = release[column].astype("string").str.strip().fillna("Missing").astype(str)

release_transform = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), release_categorical),
    ("numeric", make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                              StandardScaler()), release_numeric),
], sparse_threshold=1.0)

def release_pipeline(estimator):
    return make_pipeline(clone(release_transform), estimator)

def native_cat(depth, iterations, regularization):
    return CatBoostClassifier(
        cat_features=tuple(release_categorical), depth=depth, iterations=iterations,
        l2_leaf_reg=regularization, learning_rate=0.035, bootstrap_type="Bayesian",
        bagging_temperature=0.8, random_seed=SEED, thread_count=THREADS,
        allow_writing_files=False, verbose=False,
    )

experiments = {
    "Native CatBoost, depth 4": native_cat(4, 750, 6),
    "Native CatBoost, depth 6": native_cat(6, 650, 12),
    "Regularized XGBoost": release_pipeline(XGBClassifier(
        n_estimators=600, max_depth=3, learning_rate=0.03, min_child_weight=12,
        reg_alpha=0.4, reg_lambda=6, subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=THREADS,
    )),
    "Regularized LightGBM": release_pipeline(LGBMClassifier(
        n_estimators=550, num_leaves=12, learning_rate=0.03, min_child_samples=70,
        reg_alpha=0.4, reg_lambda=6, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8, random_state=SEED, n_jobs=THREADS, verbosity=-1,
    )),
    "Bagged random forest": release_pipeline(RandomForestClassifier(
        n_estimators=350, max_depth=13, min_samples_leaf=15, max_features=0.5,
        bootstrap=True, max_samples=0.8, random_state=SEED, n_jobs=THREADS,
    )),
    "Ridge logistic regression": release_pipeline(LogisticRegression(
        C=0.1, max_iter=1000, random_state=SEED,
    )),
}
# Interactions use a bounded numeric representation and strong L2 shrinkage.
interaction_transform = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), release_categorical),
    ("numeric", make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
        PolynomialFeatures(degree=2, include_bias=False), StandardScaler(),
    ), release_numeric),
], sparse_threshold=1.0)
experiments["Ridge logistic with interactions"] = make_pipeline(
    interaction_transform, LogisticRegression(C=0.01, max_iter=1000, random_state=SEED),
)

# EBM receives ordinal values and native categories.
for interaction_count in (0, 10, 25):
    experiments[f"Native EBM, {interaction_count} interactions"] = ExplainableBoostingClassifier(
        feature_types=[
            "nominal" if column in release_categorical else "continuous"
            for column in release
        ],
        interactions=interaction_count, min_samples_leaf=10, learning_rate=0.02,
        random_state=SEED, n_jobs=THREADS,
    )


In [8]:
X_release_fit = release.loc[X_fit.index]
X_release_val = release.loc[X_val.index]

def validation_result(probabilities):
    cutoff = max(threshold_grid, key=lambda t: accuracy_score(y_val, probabilities >= t))
    return {
        "Threshold": cutoff,
        "Validation accuracy": accuracy_score(y_val, probabilities >= cutoff),
        "Validation ROC-AUC": roc_auc_score(y_val, probabilities),
    }

validation_probabilities = {}
for name, estimator in experiments.items():
    model = clone(estimator).fit(X_release_fit, y_fit)
    validation_probabilities[name] = model.predict_proba(X_release_val)[:, 1]
    print(f"Fitted {name}", flush=True)

validation_scores = {
    name: validation_result(probability)
    for name, probability in validation_probabilities.items()
}
def validation_rank(name):
    score = validation_scores[name]
    return score["Validation accuracy"], score["Validation ROC-AUC"]

top_three = sorted(validation_scores, key=validation_rank, reverse=True)[:3]
blend_name = "Equal blend of top three"
validation_scores[blend_name] = validation_result(np.mean([
    validation_probabilities[name] for name in top_three
], axis=0))
followup_champion = max(validation_scores, key=validation_rank)
display(pd.DataFrame.from_dict(validation_scores, orient="index").sort_values(
    ["Validation accuracy", "Validation ROC-AUC"], ascending=False,
).round(4))
print("Blend members:", ", ".join(top_three))
print("Follow-up choice:", followup_champion)


Fitted Native CatBoost, depth 4
Fitted Native CatBoost, depth 6
Fitted Regularized XGBoost
Fitted Regularized LightGBM
Fitted Bagged random forest
Fitted Ridge logistic regression
Fitted Ridge logistic with interactions
Fitted Native EBM, 0 interactions
Fitted Native EBM, 10 interactions
Fitted Native EBM, 25 interactions
                                  Threshold  Validation accuracy  Validation ROC-AUC
Equal blend of top three              0.510               0.6892              0.7326
Native EBM, 10 interactions           0.500               0.6892              0.7316
Native EBM, 25 interactions           0.500               0.6874              0.7318
Native EBM, 0 interactions            0.495               0.6869              0.7327
Ridge logistic with interactions      0.485               0.6869              0.7275
Native CatBoost, depth 6              0.505               0.6867              0.7318
Native CatBoost, depth 4              0.515               0.6847              0.7

In [9]:
# Fit the validation-selected challenger on all development records.
X_release_dev, X_release_test = release.loc[X_dev.index], release.loc[X_test.index]
if followup_champion == blend_name:
    followup_model = VotingClassifier(
        estimators=[(f"member_{i}", experiments[name]) for i, name in enumerate(top_three)],
        voting="soft",
    )
else:
    followup_model = clone(experiments[followup_champion])

followup_model.fit(X_release_dev, y_dev)
followup_cutoff = validation_scores[followup_champion]["Threshold"]
followup_probabilities = followup_model.predict_proba(X_release_test)[:, 1]
followup_predictions = followup_probabilities >= followup_cutoff

reference = results.loc[(champion, "Tuned")]
comparison = pd.DataFrame({
    f"Reference ({champion})": reference[["Accuracy", "Balanced accuracy", "Precision",
                                         "Recall", "F1", "ROC-AUC"]].astype(float),
    followup_champion: {
        "Accuracy": accuracy_score(y_test, followup_predictions),
        "Balanced accuracy": balanced_accuracy_score(y_test, followup_predictions),
        "Precision": precision_score(y_test, followup_predictions, zero_division=0),
        "Recall": recall_score(y_test, followup_predictions, zero_division=0),
        "F1": f1_score(y_test, followup_predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, followup_probabilities),
    },
})
display(comparison.round(4))
delta = 100 * (comparison.loc["Accuracy", followup_champion] - reference["Accuracy"])
print(f"Holdout accuracy change: {delta:+.2f} percentage points")
print(f"Challenger cutoff: {followup_cutoff:.3f}")


                   Reference (EBM)  Equal blend of top three
Accuracy                    0.6831                    0.6825
Balanced accuracy           0.6621                    0.6615
Precision                   0.6934                    0.6929
Recall                      0.8035                    0.8028
F1                          0.7444                    0.7438
ROC-AUC                     0.7319                    0.7345
Holdout accuracy change: -0.06 percentage points
Challenger cutoff: 0.510


## Cross-fitted encodings and model stacking

Test complementary model families on the same release-time inputs. Target encoding learns smoothed category-level outcome rates. Its training features use internal cross-fitting, and the mapping for validation or holdout data is learned only from the corresponding training records. Interaction categories use age with geography, offense, and supervision level.

Compare elastic-net logistic regression, two regularized neural networks, histogram gradient boosting, and a logistic stack. The stack trains on out-of-fold base-model predictions. Choose the challenger and threshold on validation accuracy, then score only that challenger on the previously used workbook holdout.

In [10]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import TargetEncoder
from threadpoolctl import threadpool_limits

encoded = release.copy()
age_group = release["Age_at_Release"].astype(str)
for column in ("Residence_PUMA", "Prison_Offense", "Supervision_Level_First"):
    encoded[f"{column}_by_age"] = release[column] + "|" + age_group
encode_categorical = [column for column in encoded if column not in release_numeric]

def encoded_pipeline(estimator):
    preprocess = ColumnTransformer([
        ("categories", TargetEncoder(target_type="binary", smooth="auto", cv=5,
                                     shuffle=True, random_state=SEED), encode_categorical),
        ("numeric", SimpleImputer(strategy="median", add_indicator=True), release_numeric),
    ])
    return make_pipeline(preprocess, StandardScaler(), estimator)

def neural(hidden_sizes, regularization):
    return encoded_pipeline(MLPClassifier(
        hidden_layer_sizes=hidden_sizes, alpha=regularization, batch_size=128,
        learning_rate_init=0.002, early_stopping=True, n_iter_no_change=15,
        max_iter=180, random_state=SEED,
    ))

advanced_models = {
    "Elastic net": encoded_pipeline(LogisticRegression(
        solver="saga", penalty="elasticnet", l1_ratio=0.2, C=0.05,
        max_iter=1000, random_state=SEED,
    )),
    "Neural net (32)": neural((32,), 0.1),
    "Neural net (64, 16)": neural((64, 16), 0.3),
    "Histogram gradient boosting": encoded_pipeline(HistGradientBoostingClassifier(
        learning_rate=0.04, max_leaf_nodes=12, min_samples_leaf=45,
        l2_regularization=6, max_iter=325, random_state=SEED,
    )),
    "Native CatBoost": native_cat(5, 700, 10),
}
# CatBoost uses the original categorical columns plus the new age combinations.
advanced_models["Native CatBoost"].set_params(cat_features=tuple(encode_categorical))
advanced_models["OOF stack"] = StackingClassifier(
    estimators=[
        ("linear", clone(advanced_models["Elastic net"])),
        ("neural", clone(advanced_models["Neural net (32)"])),
        ("histogram", clone(advanced_models["Histogram gradient boosting"])),
        ("catboost", clone(advanced_models["Native CatBoost"])),
    ],
    final_estimator=LogisticRegression(C=0.2, random_state=SEED),
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    stack_method="predict_proba",
)

advanced_scores = {}
with threadpool_limits(limits=THREADS):
    for name, estimator in advanced_models.items():
        model = clone(estimator).fit(encoded.loc[X_fit.index], y_fit)
        probabilities = model.predict_proba(encoded.loc[X_val.index])[:, 1]
        advanced_scores[name] = validation_result(probabilities)
        print(f"Fitted {name}", flush=True)
advanced_ranking = pd.DataFrame.from_dict(advanced_scores, orient="index").sort_values(
    ["Validation accuracy", "Validation ROC-AUC"], ascending=False,
)
advanced_champion = advanced_ranking.index[0]
display(advanced_ranking.round(4))


Fitted Elastic net
Fitted Neural net (32)
Fitted Neural net (64, 16)
Fitted Histogram gradient boosting
Fitted Native CatBoost
Fitted OOF stack
                             Threshold  Validation accuracy  Validation ROC-AUC
Native CatBoost                  0.515               0.6907              0.7346
OOF stack                        0.505               0.6889              0.7332
Neural net (32)                  0.500               0.6852              0.7234
Histogram gradient boosting      0.485               0.6847              0.7301
Neural net (64, 16)              0.480               0.6832              0.7246
Elastic net                      0.530               0.6809              0.7274


In [11]:
with threadpool_limits(limits=THREADS):
    advanced_model = clone(advanced_models[advanced_champion]).fit(encoded.loc[X_dev.index], y_dev)
    advanced_probabilities = advanced_model.predict_proba(encoded.loc[X_test.index])[:, 1]

advanced_cutoff = advanced_scores[advanced_champion]["Threshold"]
advanced_predictions = advanced_probabilities >= advanced_cutoff
advanced_accuracy = accuracy_score(y_test, advanced_predictions)
incumbent_accuracy = results.loc[(champion, "Tuned"), "Accuracy"]
print(f"Challenger: {advanced_champion} (cutoff {advanced_cutoff:.3f})")
print(f"Holdout accuracy: {advanced_accuracy:.2%} (incumbent {incumbent_accuracy:.2%})")
print(f"Change: {100 * (advanced_accuracy - incumbent_accuracy):+.2f} percentage points")
print(f"Holdout ROC-AUC: {roc_auc_score(y_test, advanced_probabilities):.4f}")


Challenger: Native CatBoost (cutoff 0.515)
Holdout accuracy: 68.36% (incumbent 68.31%)
Change: +0.05 percentage points
Holdout ROC-AUC: 0.7357
